# ASIC1a pI-code MD, part 2: the ladder build

Taner Karagol and Alper Karagol.

**Runtime > Run all, then leave it.** This notebook simulates one build, ladder,
end to end and writes its report to Drive. The other two have their own notebooks,
`ASIC1a_picode_MD_2_wt`, `_ladder` and `_compare`, and the three are independent, so
open all three in three Colab sessions and start them together. When all three have
finished, run `ASIC1a_picode_MD_3_compare`.

The three notebooks are generated from one source and differ in a single line, the
name of the build they claim. Nothing else can drift between them.

**Before it runs anything it checks all three CHARMM-GUI folders**, not just this
one, because a missing disulfide or a mismatched thermostat in any of them makes the
comparison meaningless. That check reads files and takes seconds, so it is worth
paying three times over to find out now instead of after a session of GPU time.

**If Colab disconnects, open it again and press Run all again.** Every stage checks
for its own output first, so a rerun continues from the last finished chunk instead
of starting over. The compiled GROMACS is cached to Drive, so only the first of the
three sessions pays the forty minute build; the cache is written under a temporary
name and moved into place, so the other two never read a half-written archive.

**What this notebook writes to Drive**

    results/ladder_report.json    distances, occupancies and RMSD for this build
    results/progress_ladder.json  what is finished, rewritten after every stage

## The question

The functional half of this project is finished and the answer is negative. Holding
position, charge change and side-chain volume fixed across 101 substitutions, pI-code
ladder membership changes the measured activation shift by -0.006 pH units
(permutation p = 0.895). Charge change carries the whole effect at +0.162 pH units per
charge unit.

That is a statement about a channel midpoint. It is not a statement about structure,
and the ladder's premise is structural: a near-isosteric swap is supposed to disturb
the fold less. These two notebooks put that premise to a direct test.

## Why position 246

Of the 35 charge-matched pairs in the measured set, K246 is the sharpest case against
the method, and it is not a single comparison:

| at K246 | measured shift | |
|---|---|---|
| K246Q, **on the ladder** | 0.420 | the near-isosteric move |
| K246M, off the ladder | 0.190 | ladder loses by 0.230 |
| K246A, off the ladder | 0.170 | ladder loses by 0.250 |
| K246V, off the ladder | 0.310 | ladder loses by 0.110 |

All three comparators carry the same charge change from lysine, and all three agree.
K246 is also the most buried scanned position in the protein at 1.4 percent relative
side-chain accessibility, and the closest to the acidic pocket centre at 3.7 angstrom.
Burial is the one descriptor that does predict the measured shift (rho -0.595 over 38
positions, p < 0.0001), so the position where the ladder loses hardest is also the one
where the environment should matter most.

If isostericity bought tolerance, the glutamine build should sit closer to wild type
than the methionine build. If the measurement is right, it will not.

## Why the whole trimer

In the deposited low-pH trimer 9E4F the NZ of K246 in chain A sits

    6.93 angstrom from D409 of chain B
    10.04 angstrom from E219 of chain B
    11.53 angstrom from D237 of its own chain A
    11.78 angstrom from D351 of its own chain A

K246 reaches across the subunit interface into the neighbour's pocket. A monomer
simulation would answer a different question, and both notebooks refuse a system that
is not a trimer.

## Scope, stated up front

The extracellular domain only, residues 67 to 427 of all three chains, solvated, with
no membrane and no transmembrane helices. UniProt P78348 gives TRANSMEM 50..66 and
428..458, so the cut is clean and the pocket is 25 angstrom or more from either
boundary. This buys a system of roughly 110,000 atoms instead of 200,000 and makes
three builds affordable. It costs any coupling between the pocket and the gate, which
these notebooks cannot see and do not claim to.

## Sampling, and what one session buys

The system is about 127,500 atoms. Scaling from a 50,000 atom benchmark of 5 ns in 25
minutes, expect roughly 4 to 5 ns per hour on an A100, so 20 ns is about four to five
hours plus minimisation and equilibration. One build fits one session comfortably, and
the three run in parallel.

Do not take that estimate on trust. The notebook prints the wall time for `prod_01`
after the first chunk, which gives the real rate within about half an hour.

20 ns per build is 60 ns of trajectory and, because the trimer carries three copies of
the pocket, 60 ns of pocket sampling per build over 12 distance measurements. It is
enough to detect a clear difference. It is not enough to make a null mean very much,
so if the comparison comes back flat, treat that as a reason to extend rather than as
a result: raise `NS_PER_BUILD` and press Run all again, and the run continues from the
last finished chunk.

`WALL_BUDGET_HOURS` stops the production loop cleanly before Colab ends the session, so
the last chunk is written and recorded instead of lost mid-write.

In [ ]:
#@title 1. Settings
PROJECT         = "/content/drive/MyDrive/MDs/ASIC1a_picode"  #@param {type:"string"}
NS_PER_BUILD    = 20     #@param {type:"integer"}
NS_PER_CHUNK    = 5      #@param {type:"integer"}
WALL_BUDGET_HOURS = 23.0 #@param {type:"number"}
#@markdown Stops cleanly here; Run all again to continue from the last chunk.
TARGET_FRAMES   = 2000   #@param {type:"integer"}
#@markdown Frames per build. The write interval is recomputed from this,
#@markdown since CHARMM-GUI's default is set for a 1 ns run.
BURN_FRACTION   = 0.2    #@param {type:"number"}
#@markdown Fraction of each trajectory discarded as settling before any number is taken.
CACHE_GROMACS   = True   #@param {type:"boolean"}
#@markdown Keep the compiled CUDA GROMACS on Drive so later sessions skip the build.
GMX_VER         = "2026.3"    #@param {type:"string"}
GMX_SIMD        = "AVX2_256"  #@param ["AVX2_256", "AVX_512", "SSE4.1", "AUTO"]
MAXWARN         = 10     #@param {type:"integer"}

import time
T_START = time.time()
# One build per notebook, run in parallel sessions. BUILD_ORDER stays the full
# set because the pre-flight checks all three folders before anything starts.
BUILD = "ladder"
BUILD_ORDER = ["wt", "ladder", "compare"]
assert BUILD in BUILD_ORDER
print("this notebook runs build:", BUILD.upper())
print("project      ", PROJECT)
print("sampling     ", NS_PER_BUILD, "ns")
print("wall budget  ", WALL_BUDGET_HOURS, "hours")

In [ ]:
#@title 2. Drive, GPU and GROMACS
from google.colab import drive
drive.mount('/content/drive')

import os, sys, json, glob, re, subprocess, shutil, datetime, math, collections
import numpy as np

PREFIX = "/usr/local/gromacs"
os.environ["PATH"] = PREFIX + "/bin:" + os.environ["PATH"]
RESULTS = os.path.join(PROJECT, "results")
INPUTS = os.path.join(PROJECT, "inputs")
os.makedirs(RESULTS, exist_ok=True)

SPEC_PATH = os.path.join(INPUTS, "build_spec.json")
if not os.path.isfile(SPEC_PATH):
    raise SystemExit("STOP: no %s. Run part 1 first; this notebook reads the build "
                     "spec it writes, so the two cannot disagree." % SPEC_PATH)
SPEC = json.load(open(SPEC_PATH))
POSITION = SPEC["position"]
EXPECT = SPEC["expect"]
print("build spec: position %d, %s" % (POSITION, EXPECT))

missing = [b for b in BUILD_ORDER
           if not os.path.isdir(os.path.join(PROJECT, b, "gromacs"))]
if missing:
    raise SystemExit("STOP: no CHARMM-GUI folder for %s. Expected %s/<build>/gromacs "
                     "for each of %s. Finish the CHARMM-GUI step from part 1's "
                     "checklist first." % (missing, PROJECT, BUILD_ORDER))

print("\n--- GPU ---")
# check=False catches a non-zero exit, not a missing nvidia-smi, which raises
# FileNotFoundError. A missing GPU is recorded here and refused at build start.
HAVE_GPU = False
GPU_NAME = None
try:
    _g = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                         "--format=csv,noheader"], check=False, text=True,
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    if _g.returncode == 0 and (_g.stdout or "").strip():
        HAVE_GPU = True
        GPU_NAME = _g.stdout.strip().split("\n")[0]
        print(GPU_NAME)
        print("Chunk times scale with this card, so record it with the run: the same "
              "system takes several times longer on a T4 than on an A100.")
    else:
        print("nvidia-smi exited %d, so the runtime claims a GPU it cannot use."
              % _g.returncode)
        print((_g.stdout or "").strip()[:400])
except FileNotFoundError:
    print("no nvidia-smi, so this runtime has no GPU.")
    print("That is fine for the salvage and compare notebooks: they only call "
          "trjconv, trjcat, distance and rms, all of which run on the CPU.")
    print("A production run needs one. Runtime > Change runtime type > T4 GPU.")


def sh(cmd, cwd=None, label=""):
    print("\n$ " + cmd, flush=True)
    r = subprocess.run(cmd, shell=True, cwd=cwd, text=True,
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    lines = (r.stdout or "").split("\n")
    if len(lines) > 60:
        print("\n".join(lines[:25]))
        print("   ... %d lines omitted ..." % (len(lines) - 45))
        print("\n".join(lines[-20:]))
    else:
        print(r.stdout)
    if r.returncode != 0:
        raise RuntimeError("%s FAILED (exit %d). The output above is the reason."
                           % (label or cmd, r.returncode))


def gmx_version():
    try:
        return subprocess.run(["gmx", "--version"], capture_output=True, text=True,
                              timeout=90).stdout
    except Exception:
        return ""


CACHE = os.path.join(PROJECT, "gromacs_cuda_%s.tar.gz" % GMX_VER)
CACHE_STALE = False

if "CUDA" in gmx_version():
    print("\nCUDA gmx already present in this session")
elif CACHE_GROMACS and os.path.isfile(CACHE):
    # The cached binary is tested against this session's driver, and a part-written
    # archive is tolerated; either failure falls through to a rebuild.
    print("\nrestoring the cached GROMACS from", CACHE)
    try:
        sh("tar xzf '%s' -C /" % CACHE, label="restore cache")
    except Exception as e:
        print("the cache could not be read (%s); another session may be writing it. "
              "Building from source instead." % e)
        shutil.rmtree(PREFIX, ignore_errors=True)
    if "CUDA" not in gmx_version():
        print("the cached build does not run here, rebuilding from source")
        shutil.rmtree(PREFIX, ignore_errors=True)
        # A binary built against an older CUDA stops loading, so it is replaced.
        CACHE_STALE = True

if "CUDA" not in gmx_version():
    nvcc = subprocess.run("which nvcc", shell=True, capture_output=True,
                          text=True).stdout.strip()
    if not nvcc:
        raise SystemExit("STOP: nvcc not found, so there is no CUDA toolkit to build "
                         "against. Set the runtime to GPU and press Run all again.")
    print("\nbuilding GROMACS %s. 20 to 40 minutes, once per Drive folder." % GMX_VER)
    sh("apt-get -qq update && apt-get -qq install -y cmake build-essential "
       "libfftw3-dev", label="apt")
    if not os.path.isdir("/content/gromacs-%s" % GMX_VER):
        sh("wget -q https://ftp.gromacs.org/gromacs/gromacs-%s.tar.gz && "
           "tar xzf gromacs-%s.tar.gz" % (GMX_VER, GMX_VER), cwd="/content",
           label="download")
    os.makedirs("/content/gromacs-%s/build" % GMX_VER, exist_ok=True)
    sh("cmake .. -DGMX_BUILD_OWN_FFTW=OFF -DGMX_GPU=CUDA -DGMX_SIMD=%s "
       "-DCMAKE_INSTALL_PREFIX=%s -DGMX_DOUBLE=OFF -DBUILD_TESTING=OFF"
       % (GMX_SIMD, PREFIX), cwd="/content/gromacs-%s/build" % GMX_VER, label="cmake")
    sh("make -j$(nproc) && make install", cwd="/content/gromacs-%s/build" % GMX_VER,
       label="make")
    if "CUDA" not in gmx_version():
        raise SystemExit("STOP: the build finished but gmx reports no CUDA support.")
    if CACHE_GROMACS and (CACHE_STALE or not os.path.isfile(CACHE)):
        # Session-unique name, then moved, so no reader sees a partial archive.
        print("\ncaching the build to Drive so later sessions skip it")
        tmp = "%s.%d.tmp" % (CACHE, os.getpid())
        try:
            sh("tar czf '%s' %s" % (tmp, PREFIX.lstrip("/")), cwd="/", label="cache")
            os.replace(tmp, CACHE)
        except Exception as e:
            print("could not cache the build (%s); this costs the next session a "
                  "rebuild and nothing else." % e)
            if os.path.isfile(tmp):
                os.remove(tmp)
    elif CACHE_GROMACS:
        print("\nanother session already cached a build, leaving it alone")

print("\n" + gmx_version().split("\n")[0])

In [ ]:
#@title 3. Helpers
AA = set("ALA ARG ASN ASP CYS GLN GLU GLY HIS HSD HSE HSP ILE LEU LYS MET PHE PRO "
         "SER THR TRP TYR VAL".split())
# Acidic-pocket pairs in human P78348 numbering, from the manuscript.
POCKET_PAIRS = [("D237", 237, "OD1", "D351", 351, "OD1"),
                ("E238", 238, "OE1", "D347", 347, "OD1"),
                ("E219", 219, "OE1", "D409", 409, "OD1"),
                ("E79",   79, "OE1", "E418", 418, "OE1")]
# Last side-chain heavy atom per residue, resolved per build, so a substitution
# cannot measure from an atom it no longer has.
TIP_OF = {"LYS": "NZ", "GLN": "NE2", "ASN": "ND2", "MET": "CE", "GLU": "OE1",
          "ASP": "OD1", "ALA": "CB", "VAL": "CG1", "ARG": "NH1",
          "HSD": "NE2", "HSE": "NE2"}


def gmx(*args, label="", stdin_text=None, quiet=False, check=True):
    cmd = ["gmx"] + [str(a) for a in args]
    if not quiet:
        print("$ " + " ".join(cmd), flush=True)
    r = subprocess.run(cmd, text=True, input=stdin_text,
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    out = r.stdout or ""
    lines = out.split("\n")
    if not quiet:
        if len(lines) > 80:
            print("\n".join(lines[:30]))
            print("   ... %d lines omitted ..." % (len(lines) - 60))
            print("\n".join(lines[-30:]))
        else:
            print(out)
    if r.returncode != 0 and check:
        if quiet:
            print(out[-3000:])
        raise RuntimeError("%s failed (exit %d). The output above is the reason."
                           % (label or cmd[1], r.returncode))
    return r.returncode, out


def must_exist(path, what):
    if not os.path.isfile(path):
        raise RuntimeError("%s reported success but %s was not written." % (what, path))


def read_pdb_atoms(path):
    """-> list of (serial, segid, resid, resname, atomname)."""
    out = []
    for L in open(path, errors="replace"):
        if not L.startswith(("ATOM", "HETATM")):
            continue
        seg = L[72:76].strip() or L[21].strip() or "?"
        out.append((int(L[6:11]), seg, int(L[22:26]), L[17:20].strip(),
                    L[12:16].strip()))
    return out


def protein_segments(atoms):
    """Segments holding amino acids.

    gmx does not always carry the CHARMM-GUI segid through to its output PDB, in which
    case read_pdb_atoms falls back to the chain letter and a 'PROA' prefix test
    silently finds nothing. A residue-name test is true of either spelling.
    """
    rn = {}
    for ser, seg, ri, name, an in atoms:
        rn.setdefault((seg, ri), name)
    return sorted({seg for (seg, ri), name in rn.items() if name in AA})


def progress(**kw):
    """Rewritten after every stage, so a dead session still says where it got to."""
    p = os.path.join(RESULTS, "progress_%s.json" % BUILD)
    cur = json.load(open(p)) if os.path.isfile(p) else {}
    cur.update(kw)
    cur["updated"] = datetime.datetime.now().isoformat(timespec="seconds")
    cur["elapsed_hours"] = round((time.time() - T_START) / 3600.0, 2)
    json.dump(cur, open(p, "w"), indent=2)
    return cur


def budget_left():
    return WALL_BUDGET_HOURS - (time.time() - T_START) / 3600.0


def xtc_frames(path):
    """Frame count straight from gmx check.

    An absent or unreadable file returns None, which callers treat as "re-run
    this chunk and everything after it". Output that gmx produced but this
    function cannot parse stops instead, rather than silently re-running every
    chunk of every build on a paid GPU.

    The count is read from the Coords row of gmx check's summary table, with the
    Last frame line as a second chance if that table is ever renamed.
    """
    if not os.path.isfile(path):
        return None
    r = subprocess.run([os.path.join(PREFIX, "bin", "gmx"), "check", "-f", path],
                       text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    out = r.stdout or ""
    m = re.search(r"^\s*Coords\s+(\d+)", out, re.M)
    if m:
        return int(m.group(1))
    m = re.search(r"Last frame\s+(\d+)", out)
    if m:
        return int(m.group(1)) + 1
    if r.returncode != 0:
        return None
    raise SystemExit(
        "STOP: gmx check read %s and exited 0, but its output carried neither a "
        "Coords row nor a Last frame line, so the frame count is unknown. Fix "
        "this parse rather than letting every chunk be re-run as unsound. The "
        "first lines of the output were:\n%s"
        % (os.path.basename(path), "\n".join(out.split("\n")[:12])))


def xtc_last_time(path):
    """Time stamp of the last frame, in ps. None when unreadable."""
    if not os.path.isfile(path):
        return None
    r = subprocess.run([os.path.join(PREFIX, "bin", "gmx"), "check", "-f", path],
                       text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    t = None
    for L in (r.stdout or "").split("\n"):
        m = re.search(r"frame\s+\d+\s+time\s+([-\d.eE+]+)", L)
        if m:
            t = float(m.group(1))
    return t


def want_frames(mdp, ns):
    """Frames a chunk of ns nanoseconds should hold, read from the mdp."""
    dt = float(mdp_get(mdp, "dt"))
    every = int(mdp_get(mdp, "nstxout-compressed"))
    return int(round(ns * 1000.0 / dt / every))


def ndx_args(group="SOLU"):
    """-n index.ndx, with a clear stop when it is not there.

    SOLU and SOLV are CHARMM-GUI's own groups and exist only in index.ndx. Given
    -s and no -n, trjconv and rms build the default groups from the topology,
    which hold System, Protein, the side chain sets and the residue names, and no
    SOLU. The selection then fails as "Cannot read from input" after the group
    list has scrolled past, so the failure is easy to misread. grompp is not
    affected because every grompp call here passes -n.
    """
    if not os.path.isfile("index.ndx"):
        raise SystemExit(
            "STOP: no index.ndx in %s, so the group %s cannot be resolved. It comes "
            "from the CHARMM-GUI download and is staged in with the other inputs, so "
            "if it is missing here the staging is wrong."
            % (os.getcwd(), group))
    return ("-n", "index.ndx")


def stage_in(run, work, n_chunks, per_chunk):
    """Copy inputs and any sound restart files from Drive onto local VM disk.

    Everything heavy then runs on local disk. A concatenated trajectory written
    straight into the Drive mount leaves gigabytes of upload in flight, and any
    part still unwritten when the runtime ends comes back truncated.

    A chunk is reusable only if every chunk before it is. Re-running chunk k
    produces a different trajectory from k onward, so a later chunk that came off
    the old branch cannot be stitched onto the new one.
    """
    os.makedirs(work, exist_ok=True)
    heavy = ("prod_all.xtc", "prod_nojump.xtc", "prod_fit.xtc")
    for name in sorted(os.listdir(run)):
        src = os.path.join(run, name)
        if os.path.isdir(src):
            if name == "toppar":
                shutil.copytree(src, os.path.join(work, name), dirs_exist_ok=True)
            continue
        if re.match(r"prod_\d\d\.xtc$", name) or name in heavy:
            continue
        shutil.copy2(src, os.path.join(work, name))
    kept = 0
    for i in range(1, n_chunks + 1):
        nm = "prod_%02d" % i
        src = os.path.join(run, nm + ".xtc")
        if not (os.path.isfile(src) and os.path.isfile(os.path.join(work, nm + ".gro"))):
            print("  %s absent, so it and every chunk after it is re-run" % nm)
            break
        shutil.copy2(src, os.path.join(work, nm + ".xtc"))
        got = xtc_frames(os.path.join(work, nm + ".xtc"))
        if got is None or got < per_chunk - 1:
            os.remove(os.path.join(work, nm + ".xtc"))
            print("  %s holds %s of %d frames, so it and every chunk after it is "
                  "re-run" % (nm, got, per_chunk))
            break
        kept += 1
    for i in range(kept + 1, n_chunks + 1):
        for ext in (".xtc", ".gro", ".cpt", "_prev.cpt", ".tpr", ".log", ".edr"):
            f = os.path.join(work, "prod_%02d%s" % (i, ext))
            if os.path.isfile(f):
                os.remove(f)
    print("  staged into %s, %d of %d chunks reusable" % (work, kept, n_chunks))
    return kept


DRIVE_PENDING = [0]       # bytes handed to the Drive mount and not yet waited on
DRIVE_SENT = {}           # drive path -> bytes written, for the check at the end


def stage_out(work, run, names):
    """Copy products into the Drive mount and count the bytes now in flight.

    The size check below compares the destination with the source, and on a FUSE
    mount that compares the local cache against the local file. It catches a short
    write and says nothing at all about whether Google has the bytes, so a
    truncated upload passes it. What
    gives the upload time is drive_wait, and the only real proof is the next
    session's stage_in, which counts frames in the copy it pulls back down.
    """
    for nm in names:
        src = os.path.join(work, nm)
        if not os.path.isfile(src):
            continue
        dst = os.path.join(run, nm)
        shutil.copy2(src, dst)
        got, want = os.path.getsize(dst), os.path.getsize(src)
        if got != want:
            raise SystemExit("STOP: %s is %d bytes in the Drive mount against %d on "
                             "disk, so even the cached copy is short."
                             % (nm, got, want))
        DRIVE_PENDING[0] += want
        DRIVE_SENT[dst] = want


def drive_wait(why="", floor=20.0, mbps=12.0):
    """Give the Drive daemon time to finish the uploads just handed to it.

    Colab's mount writes to a local cache and uploads in the background, so a
    runtime that ends with uploads in flight leaves truncated files behind. There
    is no way to ask Google what it holds without unmounting, so this waits on a
    throughput estimate with a floor. It is a margin, not a proof. Twenty seconds
    against an eighty minute chunk is not worth economising on.
    """
    mb = DRIVE_PENDING[0] / 1e6
    DRIVE_PENDING[0] = 0
    if mb <= 0:
        return
    secs = max(floor, mb / mbps)
    print("  waiting %.0f s for Drive to take %.0f MB%s"
          % (secs, mb, " after " + why if why else ""), flush=True)
    os.sync()
    time.sleep(secs)


def drive_verify():
    """Flush the mount, bring it back, and compare every file this run uploaded.

    This is the only check in the session that asks Google rather than the local
    cache. flush_and_unmount makes Colab finish its pending writes, and the sizes
    read after the remount come from Drive's own metadata instead of from a page
    that may never have been sent. It runs last, with every file copied and every
    report written, so if the remount ever stalls it costs an open tab and nothing
    else.
    """
    try:
        from google.colab import drive as _drive
        os.sync()
        print("\nflushing the Drive mount, then checking what Drive actually holds")
        _drive.flush_and_unmount()
        _drive.mount("/content/drive")
    except Exception as _e:
        print("\ncould not flush and remount Drive (%s), so the uploads are "
              "unverified. Leave this tab open for a couple of minutes before "
              "closing it." % _e)
        return
    bad, n = [], 0
    for path, want in sorted(DRIVE_SENT.items()):
        n += 1
        got = os.path.getsize(path) if os.path.isfile(path) else None
        if got != want:
            bad.append((os.path.basename(path), got, want))
    if bad:
        for nm, got, want in bad:
            print("   %-22s %s bytes on Drive against %d written" % (nm, got, want))
        raise SystemExit(
            "STOP: %d of %d files are short or missing on Drive after the flush, so "
            "the upload did not finish. The frames are not lost: open this notebook "
            "again and press Run all, which re-sends what is short and skips every "
            "chunk that is whole." % (len(bad), n))
    print("Drive holds all %d uploaded files at full size. Safe to close the runtime."
          % n)


print("helpers ready, MAXWARN =", MAXWARN)

In [ ]:
#@title 4. Run all three builds. This is the long cell.
def check_identity(run, build):
    PDB = read_pdb_atoms(os.path.join(run, "step3_input.pdb"))
    with open(os.path.join(run, "step3_input.gro"), errors="replace") as fh:
        fh.readline()
        n_gro = int(fh.readline())
    if len(PDB) != n_gro:
        raise SystemExit("STOP: %s step3_input.pdb has %d atoms and the .gro has %d. "
                         "They must be the same build." % (build, len(PDB), n_gro))
    RES = {}
    for ser, seg, ri, name, an in PDB:
        RES.setdefault((seg, ri), name)
    prot = protein_segments(PDB)
    if len(prot) != 3:
        raise SystemExit("STOP: %s has %d protein segments (%s), expected the three "
                         "subunits. The acidic pocket is completed across the subunit "
                         "interface and a monomer answers a different question."
                         % (build, len(prot), ",".join(prot)))
    want = EXPECT[build]
    bad = [s for s in prot if RES.get((s, POSITION)) != want]
    if bad:
        raise SystemExit("STOP: %s should carry %s at %d on every subunit; segments %s "
                         "carry %s. This is not the build you think it is, or the three "
                         "CHARMM-GUI uploads went in the wrong order."
                         % (build, want, POSITION, bad,
                            [RES.get((s, POSITION)) for s in bad]))
    print("  identity ok: %s%d on all of %s, %d atoms"
          % (want, POSITION, ",".join(prot), len(PDB)))
    return len(PDB), prot


def equilibrate(run):
    if not os.path.isfile("step4.0_minimization.tpr"):
        gmx("grompp", "-f", "step4.0_minimization.mdp", "-o",
            "step4.0_minimization.tpr", "-c", "step3_input.gro", "-r",
            "step3_input.gro", "-p", "topol.top", "-n", "index.ndx",
            "-maxwarn", MAXWARN, label="grompp minimisation")
        must_exist("step4.0_minimization.tpr", "grompp minimisation")
    if not os.path.isfile("step4.0_minimization.gro"):
        gmx("mdrun", "-v", "-deffnm", "step4.0_minimization", label="minimisation")
        must_exist("step4.0_minimization.gro", "minimisation")
    prev = "step4.0_minimization"
    stages = sorted(glob.glob("step4.?_equilibration.mdp"))
    if not stages:
        raise SystemExit("STOP: no step4.*_equilibration.mdp in %s. A CHARMM-GUI "
                         "solution build writes at least one, so the download is "
                         "incomplete." % run)
    for mdp in stages:
        name = mdp[:-4]
        if os.path.isfile(name + ".gro"):
            prev = name
            continue
        gmx("grompp", "-f", mdp, "-o", name + ".tpr", "-c", prev + ".gro", "-r",
            "step3_input.gro", "-p", "topol.top", "-n", "index.ndx",
            "-maxwarn", MAXWARN, label="grompp " + name)
        must_exist(name + ".tpr", "grompp " + name)
        gmx("mdrun", "-v", "-deffnm", name, label=name)
        must_exist(name + ".gro", name)
        prev = name
    return prev


def produce(run, build, eq_final):
    mdp = "step5_production.mdp"
    if not os.path.isfile(mdp):
        raise SystemExit("STOP: no %s in %s; the CHARMM-GUI download is incomplete"
                         % (mdp, run))
    dt = None
    for L in open(mdp):
        m = re.match(r"\s*dt\s*=\s*([\d.]+)", L)
        if m:
            dt = float(m.group(1))
    if dt is None:
        raise SystemExit("STOP: could not read dt from %s, so the chunk length is "
                         "unknown" % mdp)
    nsteps = int(round(NS_PER_CHUNK * 1000.0 / dt))
    n_chunks = int(math.ceil(float(NS_PER_BUILD) / NS_PER_CHUNK))
    per_chunk = want_frames(mdp, NS_PER_CHUNK)
    drive_run = os.path.join(PROJECT, build, "gromacs")
    prev = eq_final
    for i in range(1, n_chunks + 1):
        name = "prod_%02d" % i
        if os.path.isfile(name + ".gro"):
            got = xtc_frames(name + ".xtc")
            if got is not None and got >= per_chunk - 1:
                prev = name
                continue
        if budget_left() < 0.6:
            print("  wall budget nearly spent, stopping cleanly after %d of %d chunks"
                  % (i - 1, n_chunks))
            return prev, i - 1, n_chunks
        t0 = time.time()
        # A checkpoint with no .gro means an interrupted chunk; -cpi resumes it.
        resume = os.path.isfile(name + ".cpt") and not os.path.isfile(name + ".gro")
        if not os.path.isfile(name + ".tpr"):
            gmx("grompp", "-f", mdp, "-o", name + ".tpr", "-c", prev + ".gro", "-p",
                "topol.top", "-n", "index.ndx", "-maxwarn", MAXWARN,
                label="grompp " + name)
        must_exist(name + ".tpr", "grompp " + name)
        extra = ("-cpi", name + ".cpt") if resume else ()
        if resume:
            print("  %s was interrupted, resuming from its checkpoint" % name)
        gmx("mdrun", "-v", "-deffnm", name, "-nsteps", nsteps, *extra, label=name)
        must_exist(name + ".gro", name)
        got = xtc_frames(name + ".xtc")
        if got is None or got < per_chunk - 1:
            raise SystemExit("STOP: %s wrote %s frames and should hold %d. The chunk "
                             "is incomplete, and concatenating it would give a "
                             "trajectory shorter than it reports."
                             % (name, got, per_chunk))
        print("  %s %s done in %.1f min, %d frames"
              % (build, name, (time.time() - t0) / 60.0, got))
        # Each chunk goes to Drive as it closes, size verified, so nothing of
        # value is left only on the VM.
        stage_out(os.getcwd(), drive_run,
                  [name + ".gro", name + ".cpt", name + ".xtc"])
        drive_wait(why="chunk %d" % i)
        progress(**{build: dict(chunks_done=i, chunks_total=n_chunks)})
        prev = name
    return prev, n_chunks, n_chunks


def concatenate(run, n_chunks, per_chunk):
    # Anchored to two-digit chunk names: "prod_*.xtc" also matches prod_all.xtc,
    # which a second pass would feed back into itself.
    chunks = sorted(glob.glob("prod_[0-9][0-9].xtc"))
    if len(chunks) != n_chunks:
        raise SystemExit("STOP: %s holds %d chunk trajectories and needs %d: %s"
                         % (os.getcwd(), len(chunks), n_chunks, chunks))
    counts = [xtc_frames(c) for c in chunks]
    short = [(c, n) for c, n in zip(chunks, counts)
             if n is None or n < per_chunk - 1]
    if short:
        raise SystemExit("STOP: %s hold fewer than the %d frames a chunk should "
                         "have. Concatenating them would produce a trajectory "
                         "shorter than it claims to be." % (short, per_chunk))
    for f in ("prod_all.xtc", "prod_nojump.xtc", "prod_fit.xtc"):
        if os.path.isfile(f):
            os.remove(f)
    # Every chunk restarts its clock at zero, so -settime places each at its
    # real offset; plain -cat gives a non-monotonic axis.
    offsets = "".join("%g\n" % (k * NS_PER_CHUNK * 1000.0) for k in range(n_chunks))
    gmx("trjcat", "-f", *chunks, "-o", "prod_all.xtc", "-settime",
        stdin_text=offsets, label="trjcat")
    must_exist("prod_all.xtc", "trjcat")
    span = xtc_last_time("prod_all.xtc")
    if span is None or span < 0.9 * NS_PER_BUILD * 1000.0:
        raise SystemExit("STOP: prod_all.xtc ends at %s ps and should reach about "
                         "%g ps. The time axis is wrong, so every per-frame "
                         "measurement downstream would be mislabelled."
                         % (span, NS_PER_BUILD * 1000.0))
    got = xtc_frames("prod_all.xtc")
    if got is None or got < sum(counts) - n_chunks:
        raise SystemExit("STOP: prod_all.xtc holds %s frames and the %d chunks hold "
                         "%d between them. The concatenation is incomplete."
                         % (got, n_chunks, sum(counts)))
    print("  concatenated %d chunks into %d frames" % (n_chunks, got))
    tpr = sorted(glob.glob("prod_[0-9][0-9].tpr"))[0]
    # Periodic images first, then fit; together the pbc step is skipped.
    gmx("trjconv", "-s", tpr, "-f", "prod_all.xtc", "-o", "prod_nojump.xtc",
        *ndx_args(), "-pbc", "nojump", stdin_text="SOLU\n",
        label="trjconv nojump")
    must_exist("prod_nojump.xtc", "trjconv nojump")
    gmx("trjconv", "-s", tpr, "-f", "prod_nojump.xtc", "-o", "prod_fit.xtc",
        *ndx_args(), "-fit", "rot+trans", stdin_text="SOLU\nSOLU\n",
        label="trjconv fit")
    must_exist("prod_fit.xtc", "trjconv fit")
    gmx("trjconv", "-s", tpr, "-f", "prod_all.xtc", "-o", "prod_ref.pdb",
        *ndx_args(), "-dump", "0", stdin_text="SOLU\n", label="reference frame")
    must_exist("prod_ref.pdb", "reference frame")
    return tpr


def selections(build):
    """Atom numbers resolved by name, never typed in.

    The three builds differ in a side chain, and that alone renumbers every atom after
    it: the same measurement is a different serial in each build, and hardcoding gives
    a plausible number measured on the wrong atom with nothing to warn you.
    """
    REF = read_pdb_atoms("prod_ref.pdb")
    BY, RN = {}, {}
    for ser, seg, ri, name, an in REF:
        BY[(seg, ri, an)] = ser
        RN.setdefault((seg, ri), name)
    prot = protein_segments(REF)
    if len(prot) != 3:
        raise SystemExit("STOP: prod_ref.pdb has %d protein segments, expected 3"
                         % len(prot))

    def serial(seg, ri, an, what):
        s = BY.get((seg, ri, an))
        if s is None:
            have = sorted(a for (g, r, a) in BY if g == seg and r == ri)
            raise SystemExit("STOP: %s, no atom %s in %s %d (%s). Present: %s"
                             % (what, an, seg, ri, RN.get((seg, ri)), have))
        return s

    want = EXPECT[build]
    tip_atom = TIP_OF.get(want)
    if tip_atom is None:
        raise SystemExit("STOP: no tip atom defined for %s; add it to TIP_OF" % want)

    xyz = {}
    for L in open("prod_ref.pdb", errors="replace"):
        if L.startswith(("ATOM", "HETATM")):
            xyz[int(L[6:11])] = np.array([float(L[30:38]), float(L[38:46]),
                                          float(L[46:54])])
    # The gmx selection parser reads "|" as OR and "-" as subtraction, so group
    # names are alphanumeric; the readable form is kept beside them.
    SEL = []
    for seg in prot:
        for nm1, r1, a1, nm2, r2, a2 in POCKET_PAIRS:
            SEL.append(("%s_%s_%s" % (seg, nm1, nm2), "%s|%s-%s" % (seg, nm1, nm2),
                        serial(seg, r1, a1, "pocket pair"),
                        serial(seg, r2, a2, "pocket pair")))
    for seg in prot:
        tip = serial(seg, POSITION, tip_atom, "position %d tip" % POSITION)
        best, bestd = None, 1e9
        for other in prot:
            if other == seg:
                continue
            d = float(np.linalg.norm(
                xyz[tip] - xyz[serial(other, 409, "OD1", "D409")]))
            if d < bestd:
                best, bestd = other, d
        SEL.append(("%s_%d_%s_to_%s_D409" % (seg, POSITION, tip_atom, best),
                    "%s|%d%s-%s.D409" % (seg, POSITION, tip_atom, best),
                    tip, serial(best, 409, "OD1", "D409")))
    with open("picode_pairs.ndx", "w") as fh:
        for nm, pretty, a, b in SEL:
            fh.write("[ %s ]\n%d %d\n" % (nm, a, b))
    return SEL, tip_atom


def measure(run, build, tpr, SEL, tip_atom, natoms, suffix="", want_ns=None):
    gmx("distance", "-s", tpr, "-f", "prod_fit.xtc", "-n", "picode_pairs.ndx",
        "-select", ";".join('"%s"' % nm for nm, _, _, _ in SEL),
        "-oall", "picode_dist.xvg", "-xvg", "none", label="distance", quiet=True)
    must_exist("picode_dist.xvg", "gmx distance")
    D = np.loadtxt("picode_dist.xvg")
    if D.ndim == 1:
        D = D.reshape(1, -1)
    t, V = D[:, 0], D[:, 1:] * 10.0       # gmx writes nm, the manuscript uses angstrom
    if V.shape[1] != len(SEL):
        raise SystemExit("STOP: gmx distance returned %d columns for %d selections"
                         % (V.shape[1], len(SEL)))
    burn = int(len(t) * BURN_FRACTION)
    rows = {}
    for j, (nm, pretty, _, _) in enumerate(SEL):
        v = V[burn:, j]
        rows[pretty] = dict(mean=float(v.mean()), sd=float(v.std()),
                            p10=float(np.percentile(v, 10)),
                            p90=float(np.percentile(v, 90)),
                            closed_fraction=float((v < 4.0).mean()))
    gmx("rms", "-s", tpr, "-f", "prod_fit.xtc", "-o", "picode_rmsd.xvg", "-xvg",
        "none", *ndx_args(), stdin_text="SOLU\nSOLU\n", label="rmsd", quiet=True)
    must_exist("picode_rmsd.xvg", "gmx rms")
    rmsd = np.loadtxt("picode_rmsd.xvg")[burn:, 1] * 10.0
    out = dict(build=build, run=run, atoms=int(natoms), frames=int(len(t)),
               burn=int(burn), ns=float((t[-1] - t[0]) / 1000.0),
               tip_atom=tip_atom, distances=rows,
               rmsd_mean=float(rmsd.mean()), rmsd_sd=float(rmsd.std()),
               rmsd_high=bool(rmsd.mean() > 6.0),
               written=datetime.datetime.now().isoformat(timespec="seconds"))
    target = NS_PER_BUILD if want_ns is None else want_ns
    if out["ns"] < 0.9 * target:
        raise SystemExit("STOP: the trajectory being measured spans %.1f ns against "
                         "the %g expected. Something upstream delivered less than it "
                         "reported." % (out["ns"], target))
    json.dump(out, open(os.path.join(RESULTS, "%s_report%s.json" % (build, suffix)),
                        "w"), indent=2)
    print("  %d frames, %.1f ns, backbone RMSD %.2f +/- %.2f A"
          % (len(t), out["ns"], out["rmsd_mean"], out["rmsd_sd"]))
    if out["rmsd_mean"] > 6.0:
        print("  WARNING: mean RMSD above 6 A. Check all %d disulfides were declared "
              "in CHARMM-GUI before reading anything into the pocket distances."
              % (SPEC["disulfides_per_chain"] * len(SPEC["chains"])))
    return out


def psf_disulfides(run):
    """Every SG to SG bond in the PSF, which is the authoritative bond list.

    Counting the pairs you ticked on the CHARMM-GUI screen is not the same as
    counting what it built. A missed disulfide costs the whole trajectory and
    shows up only as an unexplained RMSD at the end, so it is read back here
    from the topology before anything runs.
    """
    path = os.path.join(run, "step3_input.psf")
    if not os.path.isfile(path):
        raise SystemExit("STOP: no %s, so the disulfides cannot be verified" % path)
    lines = open(path, errors="replace").read().splitlines()
    i = next(k for k, l in enumerate(lines) if "!NATOM" in l)
    nat = int(lines[i].split()[0])
    sg = {}
    for l in lines[i + 1:i + 1 + nat]:
        f = l.split()
        # index segid resid resname atomname atomtype charge mass 0
        if len(f) >= 5 and f[4] == "SG" and f[3] == "CYS":
            sg[int(f[0])] = (f[1], int(f[2]))
    j = next(k for k, l in enumerate(lines) if "!NBOND" in l)
    nb = int(lines[j].split()[0])
    tok = []
    for l in lines[j + 1:]:
        tok += [int(x) for x in l.split()]
        if len(tok) >= 2 * nb:
            break
    return sorted({tuple(sorted((sg[a], sg[b])))
                   for a, b in zip(tok[0::2], tok[1::2]) if a in sg and b in sg})


def mdp_get(path, key):
    for l in open(path, errors="replace"):
        l = l.split(";")[0]
        if "=" in l and l.split("=")[0].strip().lower() == key.lower():
            return l.split("=", 1)[1].strip()
    return None


def mdp_set(path, key, value):
    """Rewrite one mdp setting in place, leaving every other line untouched."""
    out, hit = [], False
    for l in open(path, errors="replace"):
        body = l.split(";")[0]
        if "=" in body and body.split("=")[0].strip().lower() == key.lower():
            out.append("%-23s = %s\n" % (key, value)); hit = True
        else:
            out.append(l)
    if not hit:
        out.append("%-23s = %s\n" % (key, value))
    open(path, "w").writelines(out)


#@markdown ---
print("=" * 72)
print("PRE-FLIGHT: the three builds must differ only in the substitution")
print("=" * 72)
pre = {}
for build in BUILD_ORDER:
    run = os.path.join(PROJECT, build, "gromacs")
    required = ["step3_input.gro", "step3_input.pdb", "step3_input.psf",
                "topol.top", "step5_production.mdp", "index.ndx"]
    miss = [f for f in required if not os.path.isfile(os.path.join(run, f))]
    if miss:
        raise SystemExit("STOP: %s is missing %s. It should be the gromacs/ folder "
                         "from inside the CHARMM-GUI download." % (run, miss))
    natoms, prot = check_identity(run, build)
    ss = psf_disulfides(run)
    perseg = collections.Counter(a[0] for a, _ in ss)
    mdp = os.path.join(run, "step5_production.mdp")
    pre[build] = dict(atoms=natoms, prot=prot, ss=len(ss), perseg=dict(perseg),
                      pairs={s: sorted(tuple(sorted((r1, r2)))
                                       for (sa, r1), (sb, r2) in ss if sa == s)
                             for s in prot},
                      ref_t=mdp_get(mdp, "ref_t"), dt=mdp_get(mdp, "dt"),
                      tc=mdp_get(mdp, "tc_grps"))
    print("  %-8s %7d atoms, %d disulfides %s, ref_t %s"
          % (build, natoms, len(ss), dict(perseg), pre[build]["ref_t"]))

# The same disulfide set in every build and subunit; one short is a different
# protein, not a different substitution.
ssc = {b: pre[b]["ss"] for b in BUILD_ORDER}
if len(set(ssc.values())) != 1:
    low = min(ssc, key=ssc.get); high = max(ssc, key=ssc.get)
    missing = {s: sorted(set(pre[high]["pairs"].get(s, []))
                         - set(pre[low]["pairs"].get(s, [])))
               for s in pre[high]["pairs"]}
    missing = {s: v for s, v in missing.items() if v}
    raise SystemExit(
        "STOP: the builds disagree on disulfides %s. Build '%s' is missing %s "
        "relative to '%s'. Rebuild it in CHARMM-GUI with those pairs ticked; a "
        "free thiol where the others have a bond is a different protein."
        % (ssc, low, missing, high))
if len({tuple(sorted(pre[b]["perseg"].values())) for b in BUILD_ORDER}) != 1:
    raise SystemExit("STOP: the disulfides are not evenly distributed over the "
                     "subunits in the same way in every build: %s"
                     % {b: pre[b]["perseg"] for b in BUILD_ORDER})
print("  disulfides agree across builds: %d, %s per subunit"
      % (ssc[BUILD_ORDER[0]], sorted(set(pre[BUILD_ORDER[0]]["perseg"].values()))))

for key in ("ref_t", "dt", "tc"):
    vals = {b: pre[b][key] for b in BUILD_ORDER}
    if len(set(vals.values())) != 1:
        raise SystemExit("STOP: the builds disagree on %s: %s. They must be "
                         "simulated identically." % (key, vals))
TEMPERATURE = pre[BUILD_ORDER[0]]["ref_t"]
print("  thermostat agrees across builds: ref_t %s K" % TEMPERATURE)

atoms = {b: pre[b]["atoms"] for b in BUILD_ORDER}
spread = max(atoms.values()) - min(atoms.values())
if spread > 3000:
    raise SystemExit("STOP: the builds differ by %d atoms (%s). Different box "
                     "sizes make a pocket distance incomparable." % (spread, atoms))
print("  atom spread %d, within tolerance" % spread)

# Write interval set from the sampling asked for, not CHARMM-GUI's 1 ns default.
for build in BUILD_ORDER:
    mdp = os.path.join(PROJECT, build, "gromacs", "step5_production.mdp")
    dt = float(mdp_get(mdp, "dt"))
    want = int(round(NS_PER_BUILD * 1000.0 / dt / TARGET_FRAMES))
    want = max(100, want - want % 100)
    have = mdp_get(mdp, "nstxout-compressed")
    if have is None or int(have) != want:
        mdp_set(mdp, "nstxout-compressed", want)
        print("  %-8s nstxout-compressed %s -> %d  (%d frames over %d ns)"
              % (build, have, want, NS_PER_BUILD * 1000.0 / dt / want, NS_PER_BUILD))
    else:
        print("  %-8s nstxout-compressed already %d" % (build, want))
progress(preflight="passed", disulfides=ssc[BUILD_ORDER[0]], ref_t=TEMPERATURE,
         atom_spread=spread)
print("\npre-flight passed, starting the runs\n")

incomplete = []
for build in [BUILD]:
    run = os.path.join(PROJECT, build, "gromacs")
    rep = os.path.join(RESULTS, "%s_report.json" % build)
    print("\n" + "=" * 72)
    print("BUILD %s   %s" % (build.upper(), run))
    print("=" * 72)
    if os.path.isfile(rep) and json.load(open(rep))["ns"] >= 0.95 * NS_PER_BUILD:
        print("  already finished at %.1f ns" % json.load(open(rep))["ns"])
        continue
    required = ["step3_input.gro", "step3_input.pdb", "topol.top",
                "step5_production.mdp", "index.ndx"]
    miss = [f for f in required if not os.path.isfile(os.path.join(run, f))]
    if miss:
        raise SystemExit("STOP: %s is missing %s. It should be the gromacs/ folder "
                         "from inside the CHARMM-GUI download." % (run, miss))
    if not globals().get("HAVE_GPU", False):
        raise SystemExit("STOP: build %s needs a GPU. Equilibration and production "
                         "both ask mdrun to offload, and section 2 found no usable "
                         "GPU, so this would fail hours from now with a message about "
                         "offload instead of about the runtime. Runtime > Change "
                         "runtime type > GPU, then run from the top." % build)
    natoms, prot = check_identity(run, build)
    n_chunks = int(math.ceil(float(NS_PER_BUILD) / NS_PER_CHUNK))
    per_chunk = want_frames(os.path.join(run, "step5_production.mdp"), NS_PER_CHUNK)
    work = os.path.join("/content/work", build)
    print("  a chunk should hold %d frames" % per_chunk)
    stage_in(run, work, n_chunks, per_chunk)
    os.chdir(work)
    eq = equilibrate(work)
    progress(**{build: dict(stage="equilibrated")})
    prev, done, total = produce(work, build, eq)
    if done == 0:
        incomplete.append((build, 0, total))
        continue
    tpr = concatenate(work, n_chunks, per_chunk)
    SEL, tip_atom = selections(build)
    measure(run, build, tpr, SEL, tip_atom, natoms)
    # The concatenation stays on the VM; the fitted trajectory, its reference
    # frame and the measurements go up.
    stage_out(work, run, ["prod_fit.xtc", "prod_ref.pdb", os.path.basename(tpr),
                          "picode_dist.xvg", "picode_rmsd.xvg", "picode_pairs.ndx"])
    drive_wait(why="the analysis files")
    print("  copied the fitted trajectory and its analysis files to Drive")
    if done < total:
        incomplete.append((build, done, total))
    progress(**{build: dict(stage="measured", chunks_done=done, chunks_total=total)})

os.chdir("/content")
drive_verify()
if incomplete:
    for b, d, t in incomplete:
        print("%s reached %d of %d chunks" % (b, d, t))
    raise SystemExit(
        "The wall budget ran out before %s reached %d ns. Nothing is lost: open this "
        "notebook again and press Run all. Finished chunks are skipped, and the cached "
        "GROMACS means the next session starts in about a minute."
        % (BUILD, NS_PER_BUILD))
print("\n%s finished. Run the other two builds in their own notebooks, then open "
      "ASIC1a_picode_MD_3_compare.ipynb." % BUILD.upper())